# 02 · Surrogate checks and PGD baseline (the go/no-go gate)

1. Check that every surrogate behaves on clean images (identity matches its
   mirror, CLIP is stable under blur, VAE latents are sane). A FAIL here means
   a preprocessing bug: fix it before anything else.
2. Run per-image PGD+EOT at 8/255 and 16/255 on 200 held-out images.

**Gate:** after `jpeg75`, the PGD rows should show clearly lower `*_cos` and
`*_ratio` than the `clean` rows. If 8/255 barely moves them, fix the loss or
the augmentations before training the generator.

In [ ]:
# ---- edit these ----
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"   # "" = copy from Drive
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # runs, data tar and (optionally) code live here
# Private repo: add a Colab secret GITHUB_TOKEN. FLUX VAE: add HF_TOKEN.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
CODE = "/content/Project-Soteria"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if REPO_URL:
    url = REPO_URL
    if secret("GITHUB_TOKEN"):
        url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
    if Path(CODE).exists():
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, url, CODE], check=True)
else:
    shutil.copytree(f"{DRIVE_ROOT}/code/Project-Soteria", CODE, dirs_exist_ok=True)

if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.chdir(CODE)
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Unpack the prepared dataset from Drive onto local disk (much faster to read).
if not Path("/content/data/prepared/manifest.csv").exists():
    Path("/content/data").mkdir(exist_ok=True)
    !tar -xf {DRIVE_ROOT}/data/prepared.tar -C /content/data
!head -3 /content/data/prepared/manifest.csv && wc -l /content/data/prepared/manifest.csv

In [ ]:
!python scripts/check_surrogates.py --config configs/pgd.yaml --out {DRIVE_ROOT}/surrogate_check
from IPython.display import Image as Show
Show(f"{DRIVE_ROOT}/surrogate_check/aligned_faces.png")

In [ ]:
!python -m soteria.pgd --config configs/pgd.yaml eps=0.03137 out_dir={DRIVE_ROOT}/pgd/eps8

In [ ]:
!python -m soteria.pgd --config configs/pgd.yaml eps=0.06275 out_dir={DRIVE_ROOT}/pgd/eps16

In [ ]:
import json, pandas as pd
for run in ("eps8", "eps16"):
    s = json.load(open(f"{DRIVE_ROOT}/pgd/{run}/summary.json"))
    print(f"== {run}: {s['sec_per_image']:.1f} s/image")
    display(pd.DataFrame(s["by_method"]).round(4))